# Lab 07 — Iris Deep-Dive — Stats + First Plot
**Statistics for Analysts Track** · Beginner · ~40 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Compute per-species means and standard deviations
2. Measure class separation with Cohen's d
3. Run one-way ANOVA and a t-based 95% confidence interval
4. Publish a pair-plot (seaborn or matplotlib fallback)

## Datasets (this folder)
- `iris.csv` — auto-download from `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-07-iris-deep-dive/lab-07-iris-deep-dive.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`iris.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-07-iris-deep-dive"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/labs/lab-07-iris-deep-dive/bundle/dataset.zip"
NEED = ["iris.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Statistics for Analysts Track: Group Stats, ANOVA, Confidence Intervals

> **Scenario:** Classic `iris.csv` (150 rows, 4 numeric features + species). Compute per-species statistics, a one-way ANOVA on petal length, a 95% CI for virginica sepal width, and publish a pair-plot.
>
> **You will learn:** `groupby.agg`, variance, F-stat intuition, Cohen’s d, confidence intervals, seaborn/matplotlib pair plots.
> **Time:** ~40 minutes. **Level:** Beginner. **Needs:** pandas + scipy + matplotlib/seaborn. **Env:** 🟢 Colab only.

### Stats mental map

| Excel / idea | Python | Output |
|---|---|---|
| AVERAGEIFS by species | `groupby.mean()` | per-species means |
| Spread | `.std(ddof=1)` | sample SD |
| “Do groups differ?” | one-way ANOVA | F, p |
| “How sure of a mean?” | t-based 95% CI | mean ± t·s/√n |
| “How separated?” | Cohen’s d | effect size |

Read each row left to right as a translation layer: the spreadsheet habit you already know, the pandas/SciPy call that replaces it, and the output you should expect. Keep the table beside you while the sections run.

---

### 1. Load data (local first, Colab fallback)

Every later number depends on loading the right table. Prefer the shipped `iris.csv`; download the seaborn mirror only when that file is missing, so a Colab run is reproducible and an offline run still works. Sanity-check shape and group balance before computing anything — a three-way comparison is only fair when each species contributes the same number of rows.

In [ ]:
import math, os
import pandas as pd
from scipy import stats

def load_iris():
    local = "iris.csv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv",
            local,
        )
    return pd.read_csv(local)

df = load_iris()
print(df.shape)    # (150, 5)
print(df["species"].value_counts())  # 50 each
print(df.head(3))


**What to notice:**
- `df.shape` is **(150, 5)** — four numeric measurement columns plus the `species` label.
- **50 rows per species**: the groups are balanced, so the three means in Section 2 rest on equally sized samples.
- The head already hints at the story — setosa rows carry visibly smaller petal values than the other species.

---

### 2. Per-species means and SDs

Excel users reach for AVERAGEIFS; pandas reaches for `groupby`. Report centre **and** spread together: a mean with no SD makes a wide, overlapping group look as tight as a narrow one. `.std()` uses `ddof=1` (the n−1 sample denominator), matching Excel's STDEV and scipy's convention.

In [ ]:
g = df.groupby("species")
means = g[["sepal_length", "sepal_width", "petal_length", "petal_width"]].mean().round(3)
sds   = g[["sepal_length", "sepal_width", "petal_length", "petal_width"]].std().round(3)
print(means)
print(sds)


Expected mean `petal_length`: **setosa 1.462 · versicolor 4.260 · virginica 5.552**.

**What to notice:**
- `petal_length` climbs across the species: **1.462 → 4.260 → 5.552** — the setosa-to-others jump is far larger than the versicolor-to-virginica one.
- The `.std()` print is the other half of the story: compare each mean gap to the within-species SD before calling it “separation”.
- Sepal columns sit close together across species, while petal columns fan out — Section 3 ranks that difference properly.

---

### 3. Which feature best separates setosa vs versicolor?

Raw mean gaps are not comparable across features — sepal length and petal width live on different scales, and each group has its own spread. Cohen’s d divides the gap by the pooled standard deviation (the shared within-group SD, weighted by both sample sizes), giving a unit-free effect size you can rank across all four features.

Cohen’s d = (mean₂ − mean₁) / pooled SD:

In [ ]:
sv = df[df["species"].isin(["setosa", "versicolor"])]
print(f"{'feature':14s} {'setosa':>8s} {'versicolor':>10s} {'gap':>8s} {'cohen_d':>8s}")
for c in ["sepal_length", "sepal_width", "petal_length", "petal_width"]:
    a = sv.loc[sv["species"] == "setosa", c]
    b = sv.loc[sv["species"] == "versicolor", c]
    pooled = math.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    d = (b.mean() - a.mean()) / pooled
    print(f"{c:14s} {a.mean():8.3f} {b.mean():10.3f} {abs(b.mean()-a.mean()):8.3f} {d:8.3f}")


Expected: **petal_length wins** (gap 2.798, |d| ≈ 7.9) — far larger than sepal measures.

**What to notice:**
- `petal_length` leads with **gap 2.798 and |d| ≈ 7.9** — setosa and versicolor petals barely overlap at all.
- `petal_width` ranks second (**|d| ≈ 6.82** per Exercise 2): separating these two species is a petal story, not a sepal story.
- The pooled SD in the denominator blends both groups’ variances, so a noisy feature is penalised even when its raw gap looks respectable.

> Rule of thumb: |d| > 0.8 is a “large” separation in behavioural science; 7.9 is enormous (the classes barely overlap).

---

### 4. One-way ANOVA on petal_length

Three species means invite three separate t-tests — but that multiplies the chance of a false positive (the multiple-comparisons trap) and still does not say *which* pair differs. One-way ANOVA asks one global question instead: are all three petal-length means equal? It compares variance *between* species against variance pooled *within* species; the ratio is F.

In [ ]:
groups = [df.loc[df["species"] == sp, "petal_length"].values
          for sp in df["species"].unique()]
F, p = stats.f_oneway(*groups)
print(f"F = {F:.3f}, p = {p:.3e}")
# F = 1180.161, p = 2.86e-91


H₀: all three species share the same mean petal length. p ≈ 0 → reject H₀ decisively.

F intuition: `variance_between_groups / variance_within_groups`. Huge F ⇒ between-group gaps dwarf within-group noise.

**What to notice:**
- `F = 1180.161`, `p = 2.86e-91` — p is effectively zero, so H₀ falls decisively rather than marginally.
- F is a *ratio*, not a distance: it puts signal (between-species spread) and noise (within-species spread) on the same scale.
- F alone cannot say “how big?” — pair it with Cohen’s d (Section 3) when effect size matters, not just significance.

> **Pitfall:** ANOVA assumes independent observations, approximately normal residuals inside each group, and roughly equal within-group variances (homoscedasticity). Iris is well behaved; on messy data, glance at residual plots or run Levene’s test before you trust F.

> **Pitfall:** A significant F only says *some* pair of means differs. Follow-up pairwise t-tests without correction re-open the multiple-comparisons door you just closed — use Tukey HSD or Bonferroni-adjusted p-values (see “What to learn next”).

---

### 5. 95% CI for virginica sepal_width

A single mean hides how precisely it was estimated. Because the population SD is unknown and estimated from n = 50 rows, the correct critical value comes from the t distribution with n−1 degrees of freedom rather than the normal 1.96. The interval mean ± t·s/√n is the range of mean values still plausible under the model; in repeated sampling, 95% of such intervals capture the true virginica mean.

In [ ]:
x = df.loc[df["species"] == "virginica", "sepal_width"]
n, mean, sd = len(x), x.mean(), x.std(ddof=1)
se = sd / math.sqrt(n)
t = stats.t.ppf(0.975, df=n - 1)
ci = (mean - t * se, mean + t * se)
print(f"n={n} mean={mean:.4f} sd={sd:.4f} t={t:.4f}")
print(f"95% CI = ({ci[0]:.4f}, {ci[1]:.4f})")
# n=50 mean=2.9740 sd=0.3225 t=2.0096
# 95% CI = (2.8823, 3.0657)


**What to notice:**
- Inputs: **n = 50**, mean **2.9740**, sd **0.3225**, t **2.0096** at df = 49 — a shade wider than a z-based 1.96 interval.
- **95% CI = (2.8823, 3.0657)**, width **0.1834** — the Follow-up check value.
- The interval is about the *population mean*, not about where individual virginica flowers fall; single flowers scatter with sd ≈ 0.3225 around it.

---

### 6. Pair plot (seaborn or matplotlib fallback)

Numbers can hide structure a picture reveals instantly. A pair plot draws every feature against every other feature (histograms on the diagonal), points colored by species, so separation and correlation show up in one glance. Seaborn’s `pairplot` is a one-liner; the matplotlib branch keeps the lab green when seaborn is not installed.

In [ ]:
try:
    import seaborn as sns
    import matplotlib.pyplot as plt
    sns.pairplot(df, hue="species", corner=True)
    plt.savefig("iris_pairplot.png", dpi=120)
    print("saved iris_pairplot.png (seaborn)")
except ImportError:
    import matplotlib.pyplot as plt
    feats = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
    colors = {"setosa": "tab:red", "versicolor": "tab:blue", "virginica": "tab:green"}
    fig, axes = plt.subplots(4, 4, figsize=(9, 9))
    for i, ri in enumerate(feats):
        for j, ci_ in enumerate(feats):
            ax = axes[i, j]
            if i == j:
                for sp, col in colors.items():
                    ax.hist(df.loc[df.species == sp, ri], alpha=0.5, label=sp, color=col)
            else:
                for sp, col in colors.items():
                    sub = df[df.species == sp]
                    ax.scatter(sub[ci_], sub[ri], s=8, alpha=0.7, color=col)
            if i == 3: ax.set_xlabel(ci_, fontsize=8)
            if j == 0: ax.set_ylabel(ri, fontsize=8)
    fig.tight_layout(); fig.savefig("iris_pairplot.png", dpi=120)
    print("saved iris_pairplot.png (matplotlib)")


**What to notice:**
- **petal_length vs petal_width** cleanly splits setosa from the other two species — the same fact Cohen’s d quantified in Section 3.
- The sepal panels overlap heavily, matching the much smaller |d| values for those features.
- Both branches save `iris_pairplot.png`, so the artifact check is identical with or without seaborn.

---

## Exercises (do these!)

### Exercise 1 — Mean petal_length by species
Print mean `petal_length` for each species (3 d.p.).
*Expected: setosa 1.462 · versicolor 4.260 · virginica 5.552.*

**Follow-up:** How far apart are virginica and setosa mean petal lengths? Check: 4.09.

<details>
<summary>Hint</summary>

`df.groupby("species")["petal_length"].mean().round(3)`
</details>

### Exercise 2 — Best separator setosa vs versicolor
For each of the 4 features, compute Cohen’s d between setosa and versicolor. Which feature has the largest |d|?
*Expected: petal_length (|d| ≈ 7.90); petal_width second (≈ 6.82).*

**Follow-up:** How wide is the virginica sepal-width CI? Check: 0.1834.

<details>
<summary>Hint</summary>

Pooled SD formula in Section 3; compare absolute values.
</details>

### Exercise 3 — 95% CI for virginica sepal_width
Using t with df = n−1, report mean and CI to 4 d.p.
*Expected: mean 2.9740 · 95% CI (2.8823, 3.0657).*

**Follow-up:** Re-run the ANOVA yourself — how extreme is F? Check: F about 1180.16, p below 1e-50.

<details>
<summary>Hint</summary>

`mean ± t.ppf(0.975, n-1) * std(ddof=1) / sqrt(n)`.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
print(df.groupby("species")["petal_length"].mean().round(3))
# setosa      1.462
# versicolor  4.260
# virginica   5.552

# --- Solution 2 ---
sv = df[df.species.isin(["setosa", "versicolor"])]
best, best_d = None, 0
for c in ["sepal_length", "sepal_width", "petal_length", "petal_width"]:
    a = sv.loc[sv.species == "setosa", c]
    b = sv.loc[sv.species == "versicolor", c]
    pooled = math.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    d = abs((b.mean()-a.mean())/pooled)
    if d > best_d:
        best, best_d = c, d
print(best, round(best_d, 3))  # petal_length 7.899

# --- Solution 3 ---
x = df.loc[df.species == "virginica", "sepal_width"]
n = len(x); m = x.mean(); s = x.std(ddof=1)
t = stats.t.ppf(0.975, n-1)
lo, hi = m - t*s/math.sqrt(n), m + t*s/math.sqrt(n)
print(f"{m:.4f} ({lo:.4f}, {hi:.4f})")  # 2.9740 (2.8823, 3.0657)

# --- Follow-up 1 ---
g = df.groupby("species")["petal_length"].mean()
gap = round(float(g["virginica"] - g["setosa"]), 3)
print(gap)  # 4.09
assert abs(gap - 4.09) < 0.005

# --- Follow-up 2 ---
import numpy as _np
from scipy import stats as _ss
xv = df.loc[df.species == "virginica", "sepal_width"]
nn = len(xv)
mm = xv.mean()
sse = xv.std(ddof=1) / _np.sqrt(nn)
lo, hi = _ss.t.interval(0.95, nn - 1, loc=mm, scale=sse)
w = round(round(hi, 4) - round(lo, 4), 4)
print(round(mm, 4), (round(lo, 4), round(hi, 4)), w)
assert (round(lo, 4), round(hi, 4)) == (2.8823, 3.0657) and w == 0.1834

# --- Follow-up 3 ---
from scipy import stats as _ss
grps = [df.loc[df.species == s, "petal_length"] for s in ["setosa", "versicolor", "virginica"]]
Fv, pv = _ss.f_oneway(*grps)
print(round(Fv, 2))  # ~1180.16
assert round(Fv, 2) == 1180.16 and pv < 1e-50


### What to learn next
- Kruskal–Wallis (non-parametric alternative to ANOVA).
- Pairwise t-tests with Bonferroni correction.
- Effect sizes beyond d (η² from ANOVA).
- Then: logistic regression predicting species from measurements (Course 2).
- Cheat sheet: groupby → mean/sd → ANOVA for ≥3 groups → t-CI for one mean → plot to verify.

*Files in this folder: `iris.csv` · output `iris_pairplot.png`. Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
